In [1]:
import pandas  as pd
import numpy as np

In [2]:
df = pd.read_csv('course_lead_scoring.csv')

In [3]:
df.dtypes

lead_source                     str
industry                        str
number_of_courses_viewed      int64
annual_income               float64
employment_status               str
location                        str
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [4]:
df.describe

<bound method NDFrame.describe of          lead_source       industry  number_of_courses_viewed  annual_income  \
0           paid_ads            NaN                         1        79450.0   
1       social_media         retail                         1        46992.0   
2             events     healthcare                         5        78796.0   
3           paid_ads         retail                         2        83843.0   
4           referral      education                         3        85012.0   
...              ...            ...                       ...            ...   
1457        referral  manufacturing                         1            NaN   
1458        referral     technology                         3        65259.0   
1459        paid_ads     technology                         1        45688.0   
1460        referral            NaN                         5        71016.0   
1461  organic_search        finance                         3        92855.0   

     

In [5]:
df 

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1
...,...,...,...,...,...,...,...,...,...
1457,referral,manufacturing,1,NaN,self_employed,north_america,4,0.53,1
1458,referral,technology,3,65259.0,student,europe,2,0.24,1
1459,paid_ads,technology,1,45688.0,student,north_america,3,0.02,1
1460,referral,NaN,5,71016.0,self_employed,north_america,0,0.25,1


# Data preparation

In [6]:
# 检查数据集中的所有空值情况
print("=" * 50)  
print("快速检查:")
print("=" * 50)
print(df.isnull().sum())

快速检查:
lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64


In [7]:
df.dtypes

lead_source                     str
industry                        str
number_of_courses_viewed      int64
annual_income               float64
employment_status               str
location                        str
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [11]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']

In [12]:
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

In [13]:
for col in categorical:
    df[col] = df[col].fillna('NA')

In [15]:
for col in numerical:
    df[col] = df[col].fillna(0.0)

In [16]:
# 检查数据集中的所有空值情况
print("=" * 50)  
print("快速检查:")
print("=" * 50)
print(df.isnull().sum())

快速检查:
lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


# Question 1


In [22]:
df['industry'].value_counts()

industry
retail           203
finance          200
other            198
healthcare       187
education        187
technology       179
manufacturing    174
NA               134
Name: count, dtype: int64

# Question 2

In [25]:
# 2. 计算相关性矩阵
corr_matrix = df[numerical].corr()
print(corr_matrix)

                          number_of_courses_viewed  annual_income  \
number_of_courses_viewed                  1.000000       0.009770   
annual_income                             0.009770       1.000000   
interaction_count                        -0.023565       0.027036   
lead_score                               -0.004879       0.015610   

                          interaction_count  lead_score  
number_of_courses_viewed          -0.023565   -0.004879  
annual_income                      0.027036    0.015610  
interaction_count                  1.000000    0.009888  
lead_score                         0.009888    1.000000  


In [27]:
# 获取上三角矩阵（排除对角线）
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# 堆叠并找出最大值
max_corr = upper_tri.stack().max()
max_pair = upper_tri.stack().idxmax()
print(f"\n相关性最大的特征对：{max_pair[0]} 和 {max_pair[1]}，相关系数 = {max_corr:.4f}")


相关性最大的特征对：annual_income 和 interaction_count，相关系数 = 0.0270


# Split the data

In [33]:
from sklearn.model_selection import train_test_split  #导入库  

In [34]:
df_full_train,df_test = train_test_split(df, test_size=0.2, random_state=42) 

In [35]:
len(df_full_train),len(df_test)

(1169, 293)

In [36]:
df_train,df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [37]:
len(df_full_train),len(df_test),len(df_train),len(df_val)

(1169, 293, 876, 293)

In [38]:
df_train = df_train.reset_index(drop=True)   #将上面划分数据打乱的索引，重建索引，并删除旧索引
df_val = df_val.reset_index(drop=True) 
df_test = df_test.reset_index(drop=True) 

In [40]:
# 求解三部分数据集的y值，numpy类型
y_train = df_train.converted.values 
y_val = df_val.converted.values 
y_test = df_test.converted.values 

In [42]:
# 删除三部分数据集的y值,防止y值在数据集被用作训练验证和测试时，被意外使用
del df_train['converted']
del df_val['converted']
del df_test['converted']

# Question 3

In [46]:
from sklearn.metrics import mutual_info_score

In [47]:
def  mutual_info_churn_score(series):
    return mutual_info_score(series,df_full_train.converted)

In [48]:
mi = df_full_train[categorical].apply(mutual_info_churn_score) 
mi.sort_values(ascending=False)

lead_source          0.025665
employment_status    0.013258
industry             0.011685
location             0.002253
dtype: float64

# Question 4

In [51]:
from sklearn.feature_extraction  import DictVectorizer   #导入字典向量 库

In [52]:
dv = DictVectorizer(sparse=False)   # 初始化字典向量化器,不适用稀疏矩阵，如果这个参数不加，默认结果是稀疏矩阵类型的。

In [53]:
train_dicts = df_train[categorical + numerical].to_dict(orient='records')

In [54]:
train_dicts[0] 

{'lead_source': 'organic_search',
 'industry': 'retail',
 'employment_status': 'student',
 'location': 'asia',
 'number_of_courses_viewed': 1,
 'annual_income': 0.0,
 'interaction_count': 4,
 'lead_score': 0.47}

In [55]:
dv.fit(train_dicts)

,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",False
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True


In [57]:
feature_names_out = dv.get_feature_names_out()
feature_names_out

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=other',
       'industry=retail', 'industry=technology', 'interaction_count',
       'lead_score', 'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=australia',
       'location=europe', 'location=middle_east',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [59]:
X_train =  dv.transform(train_dicts)
X_train

array([[0.0000e+00, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        1.0000e+00],
       [6.8218e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        1.0000e+00],
       [5.0616e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        1.0000e+00],
       ...,
       [5.1633e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00],
       [4.8700e+04, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        3.0000e+00],
       [0.0000e+00, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 0.0000e+00,
        2.0000e+00]], shape=(876, 31))

In [66]:
# 对验证数据集和测试数据集 做数据准备工作
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

In [67]:
X_val = dv.transform(val_dicts)

In [49]:
from sklearn.linear_model import LogisticRegression

In [60]:
# solver='liblinear'   作用：选择用于训练模型的优化算法。小数据集、二分类	速度快，支持L1/L2正则化，但不能处理多分类（ovr）,对二分类问题效果很好
# C=1.0  作用：正则化强度的倒数。默认值，平衡状态	一般是合理的起点
# max_iter=1000  作用：优化算法最大迭代次数。
# random_state=42  作用：设置随机种子，保证结果可复现。
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42) 

In [61]:
model.fit(X_train,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [62]:
model.coef_[0].round(3) #查看权重值

array([-0.   , -0.014,  0.073, -0.005, -0.004, -0.27 , -0.075,  0.106,
       -0.069, -0.031, -0.02 , -0.039, -0.058, -0.035,  0.419,  0.077,
        0.004, -0.062, -0.031, -0.229,  0.138, -0.041,  0.004, -0.05 ,
       -0.033, -0.005, -0.016, -0.006, -0.063, -0.052,  0.738])

In [64]:
model.intercept_[0]    #查看截距值

np.float64(-0.22134823572314358)

In [75]:
y_val_pred = model.predict_proba(X_val)[:,1].round(2)
y_val_pred

array([0.87, 0.84, 0.91, 0.82, 0.95, 0.52, 0.65, 0.3 , 0.67, 0.9 , 0.56,
       0.83, 0.5 , 0.97, 0.65, 0.63, 0.78, 0.95, 0.98, 0.62, 0.75, 0.24,
       0.37, 0.53, 0.86, 0.57, 0.64, 0.72, 0.78, 0.72, 0.89, 0.89, 0.9 ,
       0.8 , 0.59, 0.67, 0.36, 0.87, 0.63, 0.75, 0.89, 0.99, 0.59, 0.77,
       0.98, 0.42, 0.8 , 0.41, 0.68, 0.95, 0.65, 0.78, 0.67, 0.49, 0.34,
       0.54, 0.62, 0.98, 0.17, 0.29, 0.43, 0.37, 0.7 , 0.68, 0.59, 0.93,
       0.96, 0.65, 0.83, 0.96, 0.87, 0.53, 0.98, 0.31, 0.8 , 0.97, 0.29,
       0.69, 0.32, 0.25, 0.89, 0.62, 0.11, 0.81, 0.49, 0.86, 0.58, 0.26,
       0.65, 0.97, 0.93, 0.89, 0.92, 0.95, 0.57, 0.77, 0.54, 0.46, 0.45,
       0.94, 0.58, 1.  , 0.83, 0.88, 0.14, 0.39, 0.94, 0.97, 0.57, 0.83,
       0.97, 0.37, 0.72, 0.35, 0.83, 0.54, 0.31, 0.44, 0.81, 0.75, 0.91,
       0.84, 0.79, 0.9 , 0.7 , 0.6 , 0.53, 0.51, 0.36, 0.98, 0.4 , 0.55,
       0.9 , 0.71, 0.64, 0.88, 0.47, 0.71, 0.98, 0.51, 0.54, 0.57, 0.29,
       0.96, 0.96, 0.27, 0.49, 0.72, 0.63, 0.43, 0.

In [71]:
dict(zip(dv.get_feature_names_out(),model.coef_[0].round(3)))

{'annual_income': np.float64(-0.0),
 'employment_status=NA': np.float64(-0.014),
 'employment_status=employed': np.float64(0.073),
 'employment_status=self_employed': np.float64(-0.005),
 'employment_status=student': np.float64(-0.004),
 'employment_status=unemployed': np.float64(-0.27),
 'industry=NA': np.float64(-0.075),
 'industry=education': np.float64(0.106),
 'industry=finance': np.float64(-0.069),
 'industry=healthcare': np.float64(-0.031),
 'industry=manufacturing': np.float64(-0.02),
 'industry=other': np.float64(-0.039),
 'industry=retail': np.float64(-0.058),
 'industry=technology': np.float64(-0.035),
 'interaction_count': np.float64(0.419),
 'lead_score': np.float64(0.077),
 'lead_source=NA': np.float64(0.004),
 'lead_source=events': np.float64(-0.062),
 'lead_source=organic_search': np.float64(-0.031),
 'lead_source=paid_ads': np.float64(-0.229),
 'lead_source=referral': np.float64(0.138),
 'lead_source=social_media': np.float64(-0.041),
 'location=NA': np.float64(0.004),

In [93]:
converted_decision = (y_val_pred >=0.5)   # 设置的阈值 0.5 
converted_decision

array([ True,  True,  True,  True,  True,  True,  True, False,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True, False, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
       False,  True,  True,  True,  True,  True,  True,  True,  True,
       False,  True, False,  True,  True,  True,  True,  True, False,
       False,  True,  True,  True, False, False, False, False,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True, False,  True,  True, False,  True, False, False,  True,
        True, False,  True, False,  True,  True, False,  True,  True,
        True,  True,  True,  True,  True,  True,  True, False, False,
        True,  True,  True,  True,  True, False, False,  True,  True,
        True,  True,  True, False,  True, False,  True,  True, False,
       False,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [94]:
converted_decision.astype(int)

array([1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
       0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1,
       1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0,
       0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1,
       1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0,
       1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0,
       1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1,
       1, 1, 0, 1, 0, 1, 1])

In [95]:
(y_val == converted_decision).mean()

np.float64(0.7440273037542662)

# Question 5   特征消除法 - 找出最不重要的特征

In [96]:
# 1. 基线准确率（从Q4得到）
baseline_accuracy = 0.7440273037542662
print(f"基线准确率: {baseline_accuracy:.4f}")
print("=" * 60)

# 2. 定义需要测试的特征（仅限题目要求的三个）
features_to_test = {
    'industry': 'categorical',
    'employment_status': 'categorical',
    'lead_score': 'numerical'
}

# 存储结果
results = {}

# 3. 对每个特征进行消除测试
for feature_name, feature_type in features_to_test.items():
    print(f"\n测试特征: {feature_name}")
    print("-" * 40)
    
    # a. 根据特征类型，从对应的列表中移除该特征
    if feature_type == 'categorical':
        reduced_categorical = [col for col in categorical if col != feature_name]
        reduced_numerical = numerical.copy()  # 数值特征不变
    else:  # numerical
        reduced_numerical = [col for col in numerical if col != feature_name]
        reduced_categorical = categorical.copy()  # 分类特征不变
    
    # b. 重新构建训练数据和验证数据的字典
    train_dicts_reduced = df_train[reduced_categorical + reduced_numerical].to_dict(orient='records')
    val_dicts_reduced = df_val[reduced_categorical + reduced_numerical].to_dict(orient='records')
    
    # c. 重新训练 DictVectorizer
    dv_reduced = DictVectorizer(sparse=False)
    dv_reduced.fit(train_dicts_reduced)
    
    # d. 转换数据
    X_train_reduced = dv_reduced.transform(train_dicts_reduced)
    X_val_reduced = dv_reduced.transform(val_dicts_reduced)
    
    # e. 重新训练 LogisticRegression 模型
    model_reduced = LogisticRegression(
        solver='liblinear', 
        C=1.0, 
        max_iter=1000, 
        random_state=42
    )
    model_reduced.fit(X_train_reduced, y_train)
    
    # f. 预测并计算准确率
    y_val_pred_reduced = model_reduced.predict_proba(X_val_reduced)[:, 1]
    y_val_pred_binary_reduced = (y_val_pred_reduced >= 0.5)
    new_accuracy = (y_val == y_val_pred_binary_reduced).mean()
    
    # g. 计算差值
    diff = baseline_accuracy - new_accuracy
    
    # 存储结果
    results[feature_name] = {
        'accuracy': new_accuracy,
        'difference': diff
    }
    
    print(f"  移除 '{feature_name}' 后的准确率: {new_accuracy:.4f}")
    print(f"  与基线准确率的差值: {diff:.4f}")

# 4. 输出最终结果
print("\n" + "=" * 60)
print("特征重要性排序（差值越小越不重要）")
print("=" * 60)

# 按差值从小到大排序
sorted_results = sorted(results.items(), key=lambda x: x[1]['difference'])

for feature_name, result in sorted_results:
    print(f"  {feature_name:20s} | 差值: {result['difference']:.4f} | 准确率: {result['accuracy']:.4f}")

print("\n" + "=" * 60)
print(f"结论：在给定的三个特征中，差值最小的特征是 '{sorted_results[0][0]}'")

基线准确率: 0.7440

测试特征: industry
----------------------------------------
  移除 'industry' 后的准确率: 0.7338
  与基线准确率的差值: 0.0102

测试特征: employment_status
----------------------------------------
  移除 'employment_status' 后的准确率: 0.7338
  与基线准确率的差值: 0.0102

测试特征: lead_score
----------------------------------------
  移除 'lead_score' 后的准确率: 0.7406
  与基线准确率的差值: 0.0034

特征重要性排序（差值越小越不重要）
  lead_score           | 差值: 0.0034 | 准确率: 0.7406
  industry             | 差值: 0.0102 | 准确率: 0.7338
  employment_status    | 差值: 0.0102 | 准确率: 0.7338

结论：在给定的三个特征中，差值最小的特征是 'lead_score'


# Question 6

In [97]:
print("=" * 60)
print("Question 6: 寻找最佳正则化参数 C")
print("=" * 60)

# 1. 定义要测试的 C 值
C_values = [0.01, 0.1, 1, 10, 100]

# 存储结果
results_c = {}

# 2. 对每个 C 值训练模型
for C in C_values:
    print(f"\n训练模型 - C = {C}")
    print("-" * 40)
    
    # a. 使用所有特征（与 Q4 相同）
    # 注意：这里我们使用已经准备好的 X_train, X_val（来自 Q4）
    # 不需要重新做 DictVectorizer，因为特征没有变化
    
    # b. 训练 LogisticRegression 模型
    model = LogisticRegression(
        solver='liblinear', 
        C=C,                    # 使用当前测试的 C 值
        max_iter=1000, 
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # c. 在验证集上预测
    y_val_pred = model.predict_proba(X_val)[:, 1]
    y_val_pred_binary = (y_val_pred >= 0.5)
    
    # d. 计算准确率并保留 3 位小数
    accuracy = (y_val == y_val_pred_binary).mean()
    accuracy_rounded = round(accuracy, 3)
    
    # 存储结果
    results_c[C] = {
        'accuracy': accuracy,
        'accuracy_rounded': accuracy_rounded
    }
    
    print(f"  验证集准确率: {accuracy:.6f}")
    print(f"  保留 3 位小数: {accuracy_rounded:.3f}")

# 3. 找出最佳 C 值
print("\n" + "=" * 60)
print("结果汇总")
print("=" * 60)
print(f"{'C 值':<10} | {'准确率':<12} | {'保留3位'}")
print("-" * 40)

for C in C_values:
    acc = results_c[C]['accuracy_rounded']
    print(f"{C:<10} | {results_c[C]['accuracy']:.6f} | {acc:.3f}")

# 4. 找出最佳 C 值（准确率最高的）
best_C = max(results_c, key=lambda x: results_c[x]['accuracy'])
best_accuracy = results_c[best_C]['accuracy_rounded']

print("\n" + "=" * 60)
print(f"最佳 C 值: {best_C}")
print(f"对应的验证集准确率: {best_accuracy:.3f}")
print("=" * 60)

# 5. 如果有多个 C 值达到相同的最高准确率，选择最小的 C
print("\n检查是否有多个 C 值达到相同的最佳准确率...")

# 找出所有达到最高准确率的 C 值
max_accuracy = results_c[best_C]['accuracy']
best_C_list = [C for C in C_values if abs(results_c[C]['accuracy'] - max_accuracy) < 1e-10]

if len(best_C_list) > 1:
    print(f"有 {len(best_C_list)} 个 C 值达到了相同的最高准确率: {best_C_list}")
    print(f"根据题目要求，选择最小的 C: {min(best_C_list)}")
    best_C = min(best_C_list)
else:
    print(f"只有一个 C 值达到最高准确率: {best_C}")

print("\n" + "=" * 60)
print(f"最终答案: C = {best_C}")
print("=" * 60)

Question 6: 寻找最佳正则化参数 C

训练模型 - C = 0.01
----------------------------------------
  验证集准确率: 0.744027
  保留 3 位小数: 0.744

训练模型 - C = 0.1
----------------------------------------
  验证集准确率: 0.744027
  保留 3 位小数: 0.744

训练模型 - C = 1
----------------------------------------
  验证集准确率: 0.744027
  保留 3 位小数: 0.744

训练模型 - C = 10
----------------------------------------
  验证集准确率: 0.744027
  保留 3 位小数: 0.744

训练模型 - C = 100
----------------------------------------
  验证集准确率: 0.744027
  保留 3 位小数: 0.744

结果汇总
C 值        | 准确率          | 保留3位
----------------------------------------
0.01       | 0.744027 | 0.744
0.1        | 0.744027 | 0.744
1          | 0.744027 | 0.744
10         | 0.744027 | 0.744
100        | 0.744027 | 0.744

最佳 C 值: 0.01
对应的验证集准确率: 0.744

检查是否有多个 C 值达到相同的最佳准确率...
有 5 个 C 值达到了相同的最高准确率: [0.01, 0.1, 1, 10, 100]
根据题目要求，选择最小的 C: 0.01

最终答案: C = 0.01
